In [ ]:
# Import path to source directory (bit of a hack in Jupyter)
import sys
import os
pwd = %pwd
sys.path.append(os.path.join(pwd, '..'))
sys.path.append(os.path.join(pwd,'src'))

# Ensure modules are reloaded on any change (very useful when developing code on the fly)
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import seaborn as sns
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from src.implementation_heuristic import *
from src.visual import *
PREFIX = "../"



## Combination of algorithms to achieve best result


- create udp
- create chromosome
- use find chromosome to apply the method with some parameters
- !!! always call the fitness afterwards to update the cube ensemble

In [ ]:
PROBLEM = "JWST_INV"
udp = programmable_cubes_UDP(PROBLEM,"../")
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)



### Show problem

In [ ]:


udp.plot('ensemble',types)
udp.plot('target',types)
debug_plot(udp)


## Algorithm

In [ ]:
print(udp.fitness(np.array([-1])))
ci = udp.final_cube_positions
chrom = np.load("chromosome_jwst_almost.npy")
#chrom = []
print(udp.fitness(end_chromosome(chrom),ci),analyze_first_and_second_mistakes(udp))


In [ ]:
chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,1,chrom,budget=1000,verbose=True)
print(udp.fitness(end_chromosome(chrom),ci))
analyze_first_and_second_mistakes(udp)

In [ ]:
chrom = np.concatenate([chrom,find_chromosome_heuristic(udp,budget=1000,dont_care_about_color=True)])
udp.fitness(end_chromosome(chrom))

In [ ]:
chrom = np.concatenate([chrom,find_chromosome(udp,pairing=pair_colours,pathfinding=brute_force_move_to)])
udp.fitness(end_chromosome(chrom))


In [ ]:
chrom = np.concatenate([chrom,run_removal_of_stuck_cubes_on_udp(udp,5,False)])
udp.fitness(end_chromosome(chrom))

In [ ]:
analyze_first_and_second_mistakes(udp)

In [ ]:
print(udp.fitness(end_chromosome(chrom)))
tmp_chrom, success = fill_empty_space_with_type(ProgrammableCubes(udp.final_cube_positions),udp.final_cube_positions,np.array([7,11,1]),2,chrom)

print(chrom)
print(udp.fitness(end_chromosome(np.concatenate([chrom,tmp_chrom]))),
      analyze_first_and_second_mistakes(udp))





In [ ]:
save_achieved_config(udp,chrom)
save_mistakes(udp)

In [ ]:
# Check if invesion works on original udp
PROBLEM2 = "ISS"
udp2 = programmable_cubes_UDP(PROBLEM2,PREFIX)
inv_chrom = invert_chromosome(udp,chrom)
print(chrom,inv_chrom)
print(udp2.fitness(format_chromosome(udp2,inv_chrom)))
udp2.plot('ensemble',types)
udp2.plot('target',types)

In [ ]:
print(udp.fitness(end_chromosome(chrom)))

In [ ]:
np.save("jwst_just_8_mistakes",chrom)
udp.plot('ensemble', types)

In [ ]:
def save_setup_in_time(udp:programmable_cubes_UDP,chromosome:np.ndarray,frames=1):
    udp.fitness(np.array([-1]))
    num_of_moves = len(chromosome)/2
    for i in np.arange(frames):
        partial_chromosome = chromosome[:(int)(i*num_of_moves/frames)*2]
        print(len(partial_chromosome))
        udp.fitness(end_chromosome(partial_chromosome))
        np.save(f"out_jwst_{i}",udp.final_cube_positions)
print(len(chrom))

save_setup_in_time(udp,chrom)

In [ ]:
# save to vdb

# import numpy as np
# import openvdb as vdb   # alias so it looks like "openvdb"

# coords = udp.final_cube_positions
# types  = udp.initial_cube_types

# grid = vdb.FloatGrid()
# grid.name = "voxels"

# acc = grid.getAccessor()
# for (x, y, z), t in zip(coords, types):
#     acc.setValueOn((x, y, z), t)

# vdb.write("voxels.vdb", grids=[grid])